### USING MACHINE LEARNING TO PREDICT THE NBA

In this notebook we will use machine learning techniques to predict the outcome of NBA games. The model will predict if the home team wins (1) or away team wins (0) by analyzing several statistics across real NBA games from 2003-2020


In [1]:
import pandas as pd
import numpy as np
import warnings
DROP_TRAIN_OVERLAP = True

train = pd.read_csv("data/games_train.csv")
test = pd.read_csv("data/games_test.csv")
train.head()

,GAME_ID,HOME_TEAM_ID,VISITOR_TEAM_ID,SEASON,FG_PCT_home,FT_PCT_home,FG3_PCT_home,AST_home,REB_home,FG_PCT_away,FT_PCT_away,FG3_PCT_away,AST_away,REB_away,HOME_TEAM_WINS
0,20900469,1610612765,1610612741,2009,0.386,0.778,0.278,17,33,0.478,0.857,0.286,19,49,0
1,10300008,1610612756,1610612751,2003,0.333,0.781,0.133,17,43,0.464,0.679,0.250,22,47,0
2,21200126,1610612750,1610612744,2012,0.430,0.793,0.389,19,34,0.506,0.692,0.133,21,50,0
3,20700660,1610612741,1610612750,2007,0.425,0.719,0.500,18,48,0.437,0.815,0.071,16,36,1
4,21401002,1610612745,1610612753,2014,0.500,0.667,0.407,27,47,0.427,0.706,0.273,19,40,1


First we need to clean our data. There are 2 main issues:
1. There are duplicate GAME_IDs within the training data
2. There are GAME_IDs present in both sets

In [2]:
print(f"Loaded: train = {train.shape}, test = {test.shape}")

# Removing duplicate GAME_IDs in train
n_before = len(train)
n_exact = train.duplicated().sum()
train = train.drop_duplicates()
n_conflict = train.GAME_ID.duplicated().sum()
train = train.drop_duplicates(subset='GAME_ID', keep='first')
assert train.GAME_ID.is_unique

print(f"Duplicate GAME_IDs removed ({n_exact} exact + {n_conflict} near duplicate rows) "
f"({n_before} -> {len(train)})")

# Removing GAME_IDs present in both train and test
overlap_ids = set(train.GAME_ID) & set(test.GAME_ID)
print(f"{len(overlap_ids)} GAME_IDs appear in both train and test")
if DROP_TRAIN_OVERLAP:
    train = train[~train.GAME_ID.isin(overlap_ids)]
    print(f"{len(train)} rows dropped from train")
assert test.GAME_ID.is_unique

# See new results
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)
print(f"Final: train = {train.shape}, test = {test.shape}")
train.to_csv("data/games_train_cleaned.csv")
test.to_csv("data/games_test_cleaned.csv")


Loaded: train = (22120, 15), test = (2458, 14)
Duplicate GAME_IDs removed (40 exact + 2 near duplicate rows) (22120 -> 22078)
13 GAME_IDs appear in both train and test
22065 rows dropped from train
Final: train = (22065, 15), test = (2458, 14)


In [3]:
# Cleaned game files saved from the previous cell (saved with an index column)
train = pd.read_csv("data/games_train_cleaned.csv", index_col=0)
test = pd.read_csv("data/games_test_cleaned.csv", index_col=0)
details = pd.read_csv("data/games_details.csv", low_memory=False)

print(f"Loaded: train = {train.shape}, test = {test.shape}, details = {details.shape}")

# Dropping the COMMENT column
details = details.drop(columns=["COMMENT"])

# Removing duplicate player rows (same player listed twice in a game)
n_before = len(details)
details = details.drop_duplicates(subset=["GAME_ID", "PLAYER_ID"], keep="first")
assert not details.duplicated(["GAME_ID", "PLAYER_ID"]).any()

print(f"Duplicate (GAME_ID, PLAYER_ID) rows removed ({n_before} -> {len(details)})")

# Converting MIN to decimal minutes
# MIN comes in three formats: "mm:ss", plain "mm", and "mm.000000:ss"
def to_minutes(s):
    parts = s.astype("string").str.split(":", expand=True)
    mins = pd.to_numeric(parts[0], errors="coerce")
    secs = pd.to_numeric(parts[1], errors="coerce").fillna(0) if parts.shape[1] > 1 else 0
    out = (mins + secs / 60).astype("float64")
    return out.where(out > 0)   # negative (bad) values become NaN

details["MIN"] = to_minutes(details["MIN"])

# Removing players with 0 (or missing/invalid) minutes
n_before = len(details)
details = details[details["MIN"].notna()]
assert (details["MIN"] > 0).all()

print(f"Players with 0 minutes removed ({n_before} -> {len(details)})")

# Keeping only games that are in train or test (removes the 2021-22 games)
games = pd.concat([train.assign(SPLIT="train"), test.assign(SPLIT="test")], ignore_index=True)
games = games[["GAME_ID", "HOME_TEAM_ID", "VISITOR_TEAM_ID", "SEASON", "SPLIT"]]

n_before = len(details)
details = details.merge(games, on="GAME_ID", how="inner")
print(f"Rows from games not in train or test removed ({n_before} -> {len(details)})")

# Keeping only rows for the two teams that played (fixes game 10500109, which lists 3 teams)
n_before = len(details)
details = details[(details["TEAM_ID"] == details["HOME_TEAM_ID"]) |
                  (details["TEAM_ID"] == details["VISITOR_TEAM_ID"])]
assert details.groupby("GAME_ID")["TEAM_ID"].nunique().max() <= 2

print(f"Rows from teams not in the game removed ({n_before} -> {len(details)})")

# Flagging whether the player's team was the home team
details["IS_HOME"] = (details["TEAM_ID"] == details["HOME_TEAM_ID"]).astype(int)

# See new results
details = details.reset_index(drop=True)
print(f"Final: details = {details.shape}, games covered = {details.GAME_ID.nunique()} of {len(games)}")
details.to_csv("data/games_details_cleaned.csv", index=False)

Loaded: train = (22065, 15), test = (2458, 14), details = (668628, 29)
Duplicate (GAME_ID, PLAYER_ID) rows removed (668628 -> 668339)
Players with 0 minutes removed (668339 -> 558478)
Rows from games not in train or test removed (558478 -> 515086)
Rows from teams not in the game removed (515086 -> 515085)
Final: details = (515085, 33), games covered = 24523 of 24523


## Objective 1: Three Point Prediction

### Cleaning and feature strategy

The data needs to be modified again specifically for analysis of three-point shooting. The following measures are taken to get the best data:

1. Keep regular season and playoff games only.
2. Calculate game outcomes based on which teams had more total points.
3. Calculate rows with at least one three-point attempt. Rows are weighted by FG3A since higher volume shots give more data.
4. Exclude other shooting stats and plus/minus that give away results. 
5. Create two feature sets: One of all other stats without other shooting metrics (FGA, FGM, FG_PCT), and one with shooting stats.

In [4]:
warnings.filterwarnings("ignore")

df = pd.read_csv("data/games_details_cleaned.csv", low_memory=False)
df = df[df["GAME_ID"].astype(str).str[0].isin(["2", "4"])]          # regular season + playoffs

# Get game outcome from team points
team_pts = df.groupby(["GAME_ID", "IS_HOME"])["PTS"].sum().unstack()    # column 0 = away, 1 = home
games = pd.DataFrame({"HOME_WIN": (team_pts[1] > team_pts[0]).astype(int)})[team_pts[1] != team_pts[0]]
print(f"{len(games)} games, home win rate = {games['HOME_WIN'].mean():.3f}")

# Rows and features
pg = df[df["FG3A"] >= 1].copy()
pg["TARGET"] = pg["FG3M"] / pg["FG3A"]
pg["FG_PCT"] = pg["FGM"] / pg["FGA"]
pg["FT_PCT"] = (pg["FTM"] / pg["FTA"]).where(pg["FTA"] > 0, 0.0)
for p in ["F", "G", "C"]:
    pg[f"START_{p}"] = (pg["START_POSITION"] == p).astype(int)

base = ["MIN", "FGM", "FGA", "FG_PCT", "FTM", "FTA", "FT_PCT", "OREB", "DREB",
        "AST", "STL", "BLK", "TO", "PF", "IS_HOME", "START_F", "START_G", "START_C"]
feature_sets = {
    "All other stats":  base,
    "Without FG stats": [c for c in base if c not in ("FGM", "FGA", "FG_PCT")],
}
assert pg[base].notna().all().all()
print(f"{len(pg)} player-games with FG3A >= 1")

22970 games, home win rate = 0.593
290044 player-games with FG3A >= 1


### Player Three-Point Predictor

We want to predict a player's three-point percentage in a game based on their other stats. To do this, we use linear regression, random forest, and MLP. The seasons are split into train (2003-2016) and test (2017-2020).

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, r2_score

train = pg[pg["SEASON"] <= 2016].reset_index(drop=True)
test  = pg[pg["SEASON"] >= 2017].reset_index(drop=True)
y_tr, y_te, w_tr, w_te = train["TARGET"].values, test["TARGET"].values, train["FG3A"].values, test["FG3A"].values

def new_model(name):
    if name == "Linear":
        return LinearRegression()
    if name == "RandomForest":
        return RandomForestRegressor(n_estimators=60, min_samples_leaf=30, max_features=0.5,
                                     max_samples=0.3, random_state=42)
    return TransformedTargetRegressor(            # MLP sample weights need a recent scikit-learn (ran on 1.9.1)
        regressor=MLPRegressor(hidden_layer_sizes=(64, 32), batch_size=512, early_stopping=True,
                               max_iter=200, random_state=42),
        transformer=StandardScaler())

preds, stage1 = {}, []
floor = np.full(len(y_te), np.average(y_tr, weights=w_tr))
stage1.append({"Features": "-", "Model": "Mean (floor)", "wRMSE": np.sqrt(mean_squared_error(y_te, floor, sample_weight=w_te)),
               "wR2": r2_score(y_te, floor, sample_weight=w_te)})

for set_name, feats in feature_sets.items():
    sc = StandardScaler().fit(train[feats])
    X_tr, X_te = sc.transform(train[feats]), sc.transform(test[feats])
    for name in ["Linear", "RandomForest", "MLP"]:
        oof = np.zeros(len(train))
        for fit_i, hold_i in GroupKFold(3).split(X_tr, groups=train["SEASON"]):
            oof[hold_i] = new_model(name).fit(X_tr[fit_i], y_tr[fit_i], sample_weight=w_tr[fit_i]).predict(X_tr[hold_i])
        pred_te = new_model(name).fit(X_tr, y_tr, sample_weight=w_tr).predict(X_te)
        preds[(set_name, name)] = (oof, pred_te)
        stage1.append({"Features": set_name, "Model": name,
                       "wRMSE": np.sqrt(mean_squared_error(y_te, pred_te, sample_weight=w_te)),
                       "wR2": r2_score(y_te, pred_te, sample_weight=w_te)})

print("=== Stage 1: player-game 3P% (test seasons 2017-2020) ===")
pd.DataFrame(stage1).round(3)

=== Stage 1: player-game 3P% (test seasons 2017-2020) ===


,Features,Model,wRMSE,wR2
0,-,Mean (floor),0.234,-0.000
1,All other stats,Linear,0.178,0.422
2,All other stats,RandomForest,0.178,0.422
3,All other stats,MLP,0.178,0.423
4,Without FG stats,Linear,0.233,0.015
5,Without FG stats,RandomForest,0.233,0.013
6,Without FG stats,MLP,0.232,0.016
